# Notebook D v4: gold_person_signal_scores + gold_person_scores

**Run order:** After Notebook A v4 and Notebook B v4.

## Architecture

### gold_person_signal_scores (new)
Long-format view — one row per person × signal.
Joins `gold_research_person_signals` (detection booleans + context columns)
against `ref_signal_weights` (applicability flags + base scores).
Evaluates applicability per person per signal.
Exposes: `is_applicable`, `is_fired`, `base_score`, `dimension`.

**Adding a new signal in future requires:**
1. Add boolean column to Notebook B (signals view)
2. Insert one row into `ref_signal_weights` with applicability flags
3. That's it — this view picks it up automatically via the UNPIVOT

### gold_person_scores
Aggregates `gold_person_signal_scores` into absolute 0–100 scores.
Trivial SUM/CASE — no signal-specific logic here at all.

### gold_branch_scores
Unchanged in structure — aggregates gold_person_scores.

## Score formula
```
completeness_score = (comp_applicable - comp_fired) / comp_applicable * 100
evidence_score     = (evid_applicable - evid_fired) / evid_applicable * 100
story_score        = narr_fired / narr_applicable * 100
overall_score      = completeness * 0.40 + evidence * 0.60
```
100 = no applicable gaps/weaknesses detected (completeness/evidence)
100 = all applicable narrative signals firing (story)

## Notes on applicability
- SIGNAL_MISSING_BURIAL: `requires_death_recorded` flag used with implicit
  `death_year >= 1837` check (handled in the CASE WHEN in the view)
- SIGNAL_SINGLE_SOURCE_DEPENDENCE: `requires_lifespan_gte` repurposed as
  event_count proxy (>= 3 events ≈ >= 3 facts). Handled explicitly.
- LOW and VERY_LOW evidence density: both always in denominator;
  only one fires at a time (mutually exclusive at the detection level)


In [0]:

%sql
-- ============================================================
-- CELL 1: gold_person_signal_scores
--
-- One row per person × signal from ref_signal_weights.
-- Evaluates applicability flags against person context columns
-- exposed by gold_research_person_signals (v4).
--
-- is_applicable: TRUE if this signal's conditions are met for this person
-- is_fired:      TRUE if the signal boolean is TRUE in the signals view
--
-- To add a new signal:
--   1. Add detection boolean to Notebook B
--   2. Add row to ref_signal_weights with applicability flags
--   3. Add one line to the UNPIVOT IN (...) list below
--   4. Done — scoring and calibration views update automatically
-- ============================================================

CREATE OR REPLACE VIEW genealogy.gold_person_signal_scores AS

WITH

-- Unpivot the wide boolean signals view into long format.
-- One row per person per signal that appears in ref_signal_weights.
-- INCLUDE NULLS: retain FALSE values (default EXCLUDE NULLS would drop them).
-- To add a new signal: add one column name to the IN (...) list.
unpivoted AS (
  SELECT person_gedcom_id, signal_code, is_fired
  FROM genealogy.gold_research_person_signals
  UNPIVOT INCLUDE NULLS (
    is_fired FOR signal_code IN (
      SIGNAL_NO_BIRTH_RECORDED,
      SIGNAL_NO_DEATH_RECORDED,
      SIGNAL_NO_MARRIAGES,
      SIGNAL_NO_CHILDREN,
      SIGNAL_MISSING_PARENT,
      SIGNAL_MISSING_CENSUS_COVERAGE,
      SIGNAL_UNCOVERED_SOURCES,
      SIGNAL_DOCS_NOT_TRANSCRIBED,
      SIGNAL_LATE_LIFE_GAP,
      SIGNAL_EARLY_LIFE_ONLY,
      SIGNAL_CHILD_GAPS,
      SIGNAL_UNCONFIRMED_MILITARY,
      SIGNAL_MISSING_OCCUPATION,
      SIGNAL_MISSING_BURIAL,
      SIGNAL_NO_RESIDENCE,
      SIGNAL_NO_DOCUMENTS_AT_ALL,
      SIGNAL_FACT_CONFLICT,
      SIGNAL_TRANSCRIPT_ONLY_FACTS,
      SIGNAL_VERY_LOW_EVIDENCE_DENSITY,
      SIGNAL_LOW_EVIDENCE_DENSITY,
      SIGNAL_SINGLE_SOURCE_DEPENDENCE,
      SIGNAL_UNSOURCED_FAMILY_EVENTS,
      SIGNAL_INCOMPLETE_NAME,
      SIGNAL_IMPRECISE_DATES,
      SIGNAL_IMPRECISE_PLACES,
      SIGNAL_MIGRANT,
      SIGNAL_CONFIRMED_MILITARY,
      SIGNAL_MULTIPLE_SPOUSES,
      SIGNAL_YOUNG_DEATH,
      SIGNAL_LARGE_FAMILY,
      SIGNAL_NEWSPAPER_MENTION,
      SIGNAL_WILL_OR_PROBATE,
      SIGNAL_STORY_WRITTEN,
      SIGNAL_TRANSCRIPT_RICH,
      SIGNAL_VARIED_OCCUPATIONS,
      SIGNAL_POSSIBLE_RESIDENCE,
      SIGNAL_POSSIBLE_MARRIAGE,
      SIGNAL_POSSIBLE_CHILDREN
    )
  )
)

SELECT
  u.person_gedcom_id,
  u.signal_code,
  w.dimension,
  w.category,
  w.intent,
  w.base_score,
  w.reason_label,
  u.is_fired,

  -- ── Applicability evaluation ──────────────────────────────────────────
  -- Evaluates ref_signal_weights flags against person context columns
  -- from gold_research_person_signals (v4).
  -- A signal is applicable if applies_always=TRUE OR all non-NULL flags pass.
  CASE
    WHEN w.applies_always = TRUE THEN TRUE

    -- birth year range
    WHEN w.min_birth_year IS NOT NULL
      AND (s.birth_year IS NULL OR s.birth_year < w.min_birth_year) THEN FALSE
    WHEN w.max_birth_year IS NOT NULL
      AND (s.birth_year IS NULL OR s.birth_year > w.max_birth_year) THEN FALSE

    -- sex restriction
    WHEN w.sex_restriction IS NOT NULL
      AND s.sex != w.sex_restriction THEN FALSE

    -- must be expected to have died (birth_year <= 1930 AND expected_end_year < now)
    WHEN w.requires_expected_to_die = TRUE
      AND NOT (
        (s.birth_year IS NULL OR s.birth_year <= 1930)
        AND s.expected_end_year < year(current_date)
      ) THEN FALSE

    -- lifespan threshold (used by SIGNAL_NO_RESIDENCE: effective_span_years >= 20)
    WHEN w.requires_lifespan_gte IS NOT NULL
      AND COALESCE(s.effective_span_years, 0) < w.requires_lifespan_gte THEN FALSE

    -- survived past 40 (used by SIGNAL_LATE_LIFE_GAP)
    WHEN w.requires_survived_past_40 = TRUE
      AND NOT (
        s.birth_year IS NOT NULL
        AND s.expected_end_year >= s.birth_year + 40
      ) THEN FALSE

    -- working age (used by SIGNAL_MISSING_OCCUPATION: male, end_year >= birth + 18)
    WHEN w.requires_working_age = TRUE
      AND NOT (
        s.birth_year IS NOT NULL
        AND s.expected_end_year >= s.birth_year + 18
      ) THEN FALSE

    -- at least one expected census year (used by SIGNAL_MISSING_CENSUS_COVERAGE)
    WHEN w.requires_census_years = TRUE
      AND COALESCE(s.total_expected_censuses, 0) = 0 THEN FALSE

    -- post-1837 birth, death, or marriage (used by SIGNAL_IMPRECISE_DATES)
    WHEN w.requires_post_1837_event = TRUE
      AND NOT (
        COALESCE(s.birth_year, 0) >= 1837
        OR COALESCE(s.death_year, 0) >= 1837
        OR COALESCE(s.earliest_marriage_year, 0) >= 1837
      ) THEN FALSE

    -- has at least one matched document
    -- (used by SIGNAL_DOCS_NOT_TRANSCRIBED — can't have untranscribed docs if no docs)
    WHEN w.requires_has_document = TRUE
      AND s.SIGNAL_NO_DOCUMENTS_AT_ALL = TRUE THEN FALSE

    -- has at least one transcript
    -- (used by SIGNAL_FACT_CONFLICT, SIGNAL_TRANSCRIPT_ONLY_FACTS)
    WHEN w.requires_transcript = TRUE
      AND s.has_any_transcript = FALSE THEN FALSE

    -- has at least one citation in gold_source_coverage
    -- (used by SIGNAL_UNCOVERED_SOURCES)
    WHEN w.requires_citations = TRUE
      AND s.has_citations = FALSE THEN FALSE

    -- has family events — marriages or child births
    -- (used by SIGNAL_EARLY_LIFE_ONLY, SIGNAL_POSSIBLE_RESIDENCE)
    WHEN w.requires_family_events = TRUE
      AND COALESCE(s.num_marriages, 0) = 0
      AND COALESCE(s.num_child_births, 0) = 0 THEN FALSE

    -- proximity restriction (used by SIGNAL_NO_CHILDREN: proximity <= 1)
    WHEN w.requires_proximity_lte IS NOT NULL
      AND COALESCE(s.proximity, 99) > w.requires_proximity_lte THEN FALSE

    -- death recorded AND post-1837 (used by SIGNAL_MISSING_BURIAL)
    WHEN w.requires_death_recorded = TRUE
      AND (s.death_year IS NULL OR s.death_year < 1837) THEN FALSE

    -- total_facts threshold (used by SIGNAL_SINGLE_SOURCE_DEPENDENCE: total_facts >= 3)
    WHEN w.requires_total_facts_gte IS NOT NULL
      AND COALESCE(s.total_facts, 0) < w.requires_total_facts_gte THEN FALSE

    -- suppress for young deaths (effective_span_years 16-40)
    -- (used by SIGNAL_NO_MARRIAGES, SIGNAL_NO_CHILDREN, SIGNAL_MULTIPLE_SPOUSES)
    WHEN w.not_young_death = TRUE
      AND s.death_year IS NOT NULL
      AND s.effective_span_years BETWEEN 16 AND 40 THEN FALSE

    ELSE TRUE
  END AS is_applicable

FROM unpivoted u
JOIN genealogy.ref_signal_weights w
  ON w.signal_code = u.signal_code
JOIN genealogy.gold_research_person_signals s
  ON s.person_gedcom_id = u.person_gedcom_id;


In [0]:

%sql
-- ============================================================
-- CELL 2: gold_person_scores (v4)
--
-- Aggregates gold_person_signal_scores into absolute 0-100 scores.
-- No signal-specific logic here — all handled upstream.
--
-- story_ready threshold: >= 30 (provisional; calibrate using
-- notebook_calibration Cell 5 after first run).
-- ============================================================

CREATE OR REPLACE VIEW genealogy.gold_person_scores AS

WITH scores AS (
  SELECT
    person_gedcom_id,

    -- Completeness: applicable max and fired sum
    SUM(CASE WHEN dimension = 'completeness' AND is_applicable AND base_score > 0
             THEN base_score ELSE 0 END) AS comp_applicable,
    SUM(CASE WHEN dimension = 'completeness' AND is_applicable AND is_fired AND base_score > 0
             THEN base_score ELSE 0 END) AS comp_fired,

    -- Evidence: applicable max and fired sum
    SUM(CASE WHEN dimension = 'evidence' AND is_applicable AND base_score > 0
             THEN base_score ELSE 0 END) AS evid_applicable,
    SUM(CASE WHEN dimension = 'evidence' AND is_applicable AND is_fired AND base_score > 0
             THEN base_score ELSE 0 END) AS evid_fired,

    -- Narrative: applicable max and fired sum
    -- Exclude SIGNAL_STORY_WRITTEN from the positive narrative denominator
    SUM(CASE WHEN dimension = 'narrative' AND is_applicable AND base_score > 0
               AND signal_code != 'SIGNAL_STORY_WRITTEN'
             THEN base_score ELSE 0 END) AS narr_applicable,
    SUM(CASE WHEN dimension = 'narrative' AND is_applicable AND is_fired AND base_score > 0
               AND signal_code != 'SIGNAL_STORY_WRITTEN'
             THEN base_score ELSE 0 END) AS narr_fired,

    -- Story written flag (used to suppress story score)
    MAX(CASE WHEN signal_code = 'SIGNAL_STORY_WRITTEN' AND is_fired THEN TRUE ELSE FALSE END)
      AS story_written_flag

  FROM genealogy.gold_person_signal_scores
  GROUP BY person_gedcom_id
),

prox AS (
  SELECT person_id, MIN(ancestral_proximity) AS proximity
  FROM genealogy.gold_ancestral_proximity
  GROUP BY person_id
),

story AS (
  SELECT person_gedcom_id, story_written, story_title, story_doc_id
  FROM genealogy.silver_person_story_status
)

SELECT
  p.person_gedcom_id,
  p.given_name,
  p.first_name,
  p.surname,
  p.birth_year,
  p.death_year,
  b.branch,
  p.sex,

  -- Raw scores for diagnostics
  sc.comp_applicable,
  sc.comp_fired,
  sc.evid_applicable,
  sc.evid_fired,
  sc.narr_applicable,
  sc.narr_fired,

  -- 0-100 absolute scores
  ROUND((sc.comp_applicable - sc.comp_fired) / NULLIF(sc.comp_applicable, 0) * 100, 0)
    AS completeness_score,

  ROUND((sc.evid_applicable - sc.evid_fired) / NULLIF(sc.evid_applicable, 0) * 100, 0)
    AS evidence_score,

  -- Story potential: 0 if story already written
  CASE WHEN sc.story_written_flag OR COALESCE(st.story_written, FALSE)
       THEN 0
       ELSE ROUND(sc.narr_fired / NULLIF(sc.narr_applicable, 0) * 100, 0)
  END AS story_potential_score,

  -- Overall: completeness 40% + evidence 60%
  -- Story potential excluded (measures opportunity, not quality)
  ROUND(
    ROUND((sc.comp_applicable - sc.comp_fired) / NULLIF(sc.comp_applicable, 0) * 100, 0) * 0.40
    + ROUND((sc.evid_applicable - sc.evid_fired) / NULLIF(sc.evid_applicable, 0) * 100, 0) * 0.60
  , 0) AS overall_score,

  -- Story flags
  COALESCE(st.story_written, FALSE)  AS story_written,
  st.story_title,
  st.story_doc_id,

  -- story_ready: high story potential, not yet written
  -- Threshold provisional at 30 — recalibrate using notebook_calibration Cell 5
  CASE
    WHEN sc.story_written_flag OR COALESCE(st.story_written, FALSE) THEN FALSE
    WHEN ROUND(sc.narr_fired / NULLIF(sc.narr_applicable, 0) * 100, 0) >= 30 THEN TRUE
    ELSE FALSE
  END AS story_ready,

  -- Proximity
  pr.proximity AS ancestral_proximity,
  CASE
    WHEN pr.proximity = 0             THEN 'Direct Ancestor'
    WHEN pr.proximity = 1             THEN 'Close'
    WHEN pr.proximity BETWEEN 2 AND 3 THEN 'Collateral'
    ELSE                                   'Distant'
  END AS proximity_label

FROM genealogy.gold_person_life p
JOIN scores sc                          ON sc.person_gedcom_id = p.person_gedcom_id
LEFT JOIN genealogy.gold_person_branch b ON b.person_gedcom_id = p.person_gedcom_id
LEFT JOIN prox pr                       ON pr.person_id        = p.person_gedcom_id
LEFT JOIN story st                      ON st.person_gedcom_id = p.person_gedcom_id;


In [0]:

%sql
CREATE OR REPLACE VIEW genealogy.gold_branch_scores AS
SELECT
  branch,
  COUNT(*)                                                        AS total_individuals,
  ROUND(AVG(completeness_score))                                  AS avg_completeness,
  ROUND(AVG(evidence_score))                                      AS avg_evidence,
  ROUND(AVG(story_potential_score))                               AS avg_story_potential,
  ROUND(AVG(overall_score))                                       AS avg_overall,
  SUM(CASE WHEN story_written            THEN 1 ELSE 0 END)       AS stories_written,
  SUM(CASE WHEN story_ready
            AND NOT story_written        THEN 1 ELSE 0 END)       AS stories_ready,
  COUNT(CASE WHEN ancestral_proximity = 0 THEN 1 END)             AS direct_ancestors,
  ROUND(AVG(CASE WHEN ancestral_proximity = 0
                 THEN overall_score END))                         AS ancestor_avg_overall,
  MIN(completeness_score)                                         AS min_completeness,
  MIN(evidence_score)                                             AS min_evidence
FROM genealogy.gold_person_scores
WHERE branch IS NOT NULL
GROUP BY branch
ORDER BY avg_overall DESC;


In [0]:

%sql
-- Should all return 0 violations
SELECT 'comp_fired > comp_applicable'  AS check_name, COUNT(*) AS violations
FROM genealogy.gold_person_scores WHERE comp_fired > comp_applicable
UNION ALL
SELECT 'evid_fired > evid_applicable',  COUNT(*) FROM genealogy.gold_person_scores WHERE evid_fired > evid_applicable
UNION ALL
SELECT 'narr_fired > narr_applicable',  COUNT(*) FROM genealogy.gold_person_scores WHERE narr_fired > narr_applicable
UNION ALL
SELECT 'NULL completeness_score',       COUNT(*) FROM genealogy.gold_person_scores WHERE completeness_score IS NULL
UNION ALL
SELECT 'NULL evidence_score',           COUNT(*) FROM genealogy.gold_person_scores WHERE evidence_score IS NULL
UNION ALL
SELECT 'NULL story_potential_score',    COUNT(*) FROM genealogy.gold_person_scores WHERE story_potential_score IS NULL
UNION ALL
SELECT 'comp_applicable = 0',           COUNT(*) FROM genealogy.gold_person_scores WHERE comp_applicable = 0
UNION ALL
SELECT 'evid_applicable = 0',           COUNT(*) FROM genealogy.gold_person_scores WHERE evid_applicable = 0;


In [0]:

%sql
-- Score distributions
SELECT
  'completeness'    AS dimension,
  ROUND(MIN(completeness_score))                        AS min,
  ROUND(PERCENTILE(completeness_score, 0.25))           AS p25,
  ROUND(PERCENTILE(completeness_score, 0.50))           AS median,
  ROUND(PERCENTILE(completeness_score, 0.75))           AS p75,
  ROUND(MAX(completeness_score))                        AS max,
  ROUND(AVG(completeness_score), 1)                     AS mean,
  SUM(CASE WHEN completeness_score = 100 THEN 1 END)    AS at_100,
  SUM(CASE WHEN completeness_score = 0   THEN 1 END)    AS at_0
FROM genealogy.gold_person_scores
UNION ALL
SELECT 'evidence',
  ROUND(MIN(evidence_score)), ROUND(PERCENTILE(evidence_score, 0.25)),
  ROUND(PERCENTILE(evidence_score, 0.50)), ROUND(PERCENTILE(evidence_score, 0.75)),
  ROUND(MAX(evidence_score)), ROUND(AVG(evidence_score), 1),
  SUM(CASE WHEN evidence_score = 100 THEN 1 END),
  SUM(CASE WHEN evidence_score = 0   THEN 1 END)
FROM genealogy.gold_person_scores
UNION ALL
SELECT 'story_potential',
  ROUND(MIN(story_potential_score)), ROUND(PERCENTILE(story_potential_score, 0.25)),
  ROUND(PERCENTILE(story_potential_score, 0.50)), ROUND(PERCENTILE(story_potential_score, 0.75)),
  ROUND(MAX(story_potential_score)), ROUND(AVG(story_potential_score), 1),
  SUM(CASE WHEN story_potential_score = 100 THEN 1 END),
  SUM(CASE WHEN story_potential_score = 0   THEN 1 END)
FROM genealogy.gold_person_scores
ORDER BY dimension;


In [0]:

%sql
SELECT * FROM genealogy.gold_branch_scores;


In [0]:

%sql
-- Cross-join on thresholds to find the right story_ready cutoff.
-- Aim for ~50 story-ready individuals with good direct-ancestor coverage.
WITH thresholds AS (
  SELECT explode(sequence(10, 60, 5)) AS threshold
),
scores AS (
  SELECT person_gedcom_id, story_potential_score, story_written, proximity_label, branch
  FROM genealogy.gold_person_scores
  WHERE NOT COALESCE(story_written, FALSE)
)
SELECT
  t.threshold,
  COUNT(*)                                                        AS total_story_ready,
  SUM(CASE WHEN s.proximity_label = 'Direct Ancestor' THEN 1 ELSE 0 END) AS direct_ancestors,
  COUNT(DISTINCT s.branch)                                        AS branches_covered
FROM thresholds t
JOIN scores s ON s.story_potential_score >= t.threshold
GROUP BY t.threshold
ORDER BY t.threshold;
